# Amazon Reviews Data Exploration

## Initial Analysis of Kaggle Amazon Reviews Dataset

This notebook provides comprehensive exploration of the real Amazon reviews dataset from Kaggle.

**Dataset**: [Kaggle Amazon Reviews](https://www.kaggle.com/datasets/bittlingmayer/amazonreviews)  
**Project**: Berkeley ML/AI Capstone - NLP Analysis Version 2

## 1. Setup and Data Loading

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)

print("📚 Libraries loaded successfully!")

📚 Libraries loaded successfully!


In [ ]:
# Define data paths
PROJECT_ROOT = Path('../')
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'

print(f"Project root: {PROJECT_ROOT.absolute()}")
print(f"Data directory: {DATA_DIR.absolute()}")
print(f"Processed directory: {PROCESSED_DIR.absolute()}")

In [ ]:
# Check if data files exist
train_file = PROCESSED_DIR / 'train_processed.csv'
test_file = PROCESSED_DIR / 'test_processed.csv'

print("📁 Checking data files...")
if train_file.exists():
    train_size = train_file.stat().st_size / (1024*1024)  # MB
    print(f"✅ Training data found: {train_file.name} ({train_size:.1f} MB)")
else:
    print(f"❌ Training data not found: {train_file}")
    print("   Please run: python src/data_loader.py")

if test_file.exists():
    test_size = test_file.stat().st_size / (1024*1024)  # MB
    print(f"✅ Test data found: {test_file.name} ({test_size:.1f} MB)")
else:
    print(f"❌ Test data not found: {test_file}")
    print("   Please run: python src/data_loader.py")

In [ ]:
# Load the processed data
if train_file.exists() and test_file.exists():
    print("📊 Loading processed data...")
    
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    
    print(f"✅ Training data loaded: {len(train_df):,} rows")
    print(f"✅ Test data loaded: {len(test_df):,} rows")
    
    # Combine for overall analysis
    combined_df = pd.concat([train_df, test_df], ignore_index=True)
    print(f"📋 Combined dataset: {len(combined_df):,} total reviews")
    
else:
    print("❌ Data files not found. Please run data_loader.py first.")
    # Create sample data for demonstration
    train_df = pd.DataFrame({
        'label': [1, 2, 1, 2, 1],
        'text': ['Bad product', 'Great item', 'Terrible quality', 'Excellent!', 'Waste of money'],
        'sentiment': ['negative', 'positive', 'negative', 'positive', 'negative']
    })
    test_df = train_df.copy()
    combined_df = train_df.copy()
    print("📝 Using sample data for demonstration")

## 2. Dataset Overview and Basic Statistics

In [ ]:
# Dataset overview
print("📋 DATASET OVERVIEW")
print("=" * 60)
print(f"Total reviews: {len(combined_df):,}")
print(f"Training set: {len(train_df):,} reviews ({len(train_df)/len(combined_df):.1%})")
print(f"Test set: {len(test_df):,} reviews ({len(test_df)/len(combined_df):.1%})")
print(f"Columns: {list(combined_df.columns)}")
print(f"Data types: \n{combined_df.dtypes}")

In [ ]:
# Display sample data
print("🔍 SAMPLE DATA")
print("=" * 60)
display(combined_df.head())

In [ ]:
# Basic statistics
print("📊 BASIC STATISTICS")
print("=" * 60)

if 'text' in combined_df.columns:
    # Text length statistics
    text_lengths = combined_df['text'].str.len()
    
    print(f"Text Length Statistics:")
    print(f"  Mean: {text_lengths.mean():.0f} characters")
    print(f"  Median: {text_lengths.median():.0f} characters")
    print(f"  Min: {text_lengths.min():.0f} characters")
    print(f"  Max: {text_lengths.max():,.0f} characters")
    print(f"  Std: {text_lengths.std():.0f} characters")

# Sentiment distribution
if 'sentiment' in combined_df.columns:
    sentiment_counts = combined_df['sentiment'].value_counts()
    print(f"\nSentiment Distribution:")
    for sentiment, count in sentiment_counts.items():
        pct = count / len(combined_df) * 100
        print(f"  {sentiment.title()}: {count:,} reviews ({pct:.1f}%)")

# Missing values
missing_values = combined_df.isnull().sum()
if missing_values.sum() > 0:
    print(f"\nMissing Values:")
    for col, missing in missing_values[missing_values > 0].items():
        pct = missing / len(combined_df) * 100
        print(f"  {col}: {missing:,} ({pct:.1f}%)")
else:
    print(f"\n✅ No missing values found")

## 3. Data Quality Assessment

In [ ]:
# Data quality checks
print("🔍 DATA QUALITY ASSESSMENT")
print("=" * 60)

if 'text' in combined_df.columns:
    # Check for empty or very short texts
    empty_texts = combined_df['text'].isna().sum()
    very_short = (combined_df['text'].str.len() < 10).sum()
    very_long = (combined_df['text'].str.len() > 5000).sum()
    
    print(f"Text Quality Issues:")
    print(f"  Empty texts: {empty_texts:,}")
    print(f"  Very short (<10 chars): {very_short:,} ({very_short/len(combined_df):.2%})")
    print(f"  Very long (>5000 chars): {very_long:,} ({very_long/len(combined_df):.2%})")

# Check for duplicates
if 'text' in combined_df.columns:
    duplicates = combined_df.duplicated(subset=['text']).sum()
    print(f"  Duplicate reviews: {duplicates:,} ({duplicates/len(combined_df):.2%})")

# Label consistency check
if 'label' in combined_df.columns and 'sentiment' in combined_df.columns:
    label_sentiment_map = combined_df[['label', 'sentiment']].drop_duplicates()
    print(f"\nLabel-Sentiment Mapping:")
    for _, row in label_sentiment_map.iterrows():
        print(f"  Label {row['label']} → {row['sentiment']}")

## 4. Text Analysis and Visualization

In [ ]:
# Text length distribution
if 'text' in combined_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Overall text length distribution
    text_lengths = combined_df['text'].str.len()
    axes[0].hist(text_lengths, bins=50, alpha=0.7, edgecolor='black')
    axes[0].set_xlabel('Text Length (characters)')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Review Text Lengths')
    axes[0].grid(alpha=0.3)
    
    # Text length by sentiment
    if 'sentiment' in combined_df.columns:
        for sentiment in combined_df['sentiment'].unique():
            sentiment_texts = combined_df[combined_df['sentiment'] == sentiment]['text'].str.len()
            axes[1].hist(sentiment_texts, bins=30, alpha=0.6, label=sentiment.title(), edgecolor='black')
        
        axes[1].set_xlabel('Text Length (characters)')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title('Text Length Distribution by Sentiment')
        axes[1].legend()
        axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("📊 Text column not available for visualization")

In [ ]:
# Sentiment distribution visualization
if 'sentiment' in combined_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Overall sentiment distribution (pie chart)
    sentiment_counts = combined_df['sentiment'].value_counts()
    colors = ['#ff9999', '#66b3ff']
    axes[0].pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%', 
                colors=colors, startangle=90)
    axes[0].set_title('Overall Sentiment Distribution')
    
    # Sentiment by dataset split
    if 'split' in combined_df.columns:
        sentiment_by_split = pd.crosstab(combined_df['split'], combined_df['sentiment'], normalize='index') * 100
        sentiment_by_split.plot(kind='bar', ax=axes[1], color=colors)
        axes[1].set_xlabel('Dataset Split')
        axes[1].set_ylabel('Percentage')
        axes[1].set_title('Sentiment Distribution by Dataset Split')
        axes[1].legend(title='Sentiment')
        axes[1].tick_params(axis='x', rotation=0)
        axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("📊 Sentiment column not available for visualization")

## 5. Sample Review Analysis

In [ ]:
# Show sample reviews for each sentiment
print("📝 SAMPLE REVIEWS BY SENTIMENT")
print("=" * 60)

if 'sentiment' in combined_df.columns and 'text' in combined_df.columns:
    for sentiment in combined_df['sentiment'].unique():
        sentiment_reviews = combined_df[combined_df['sentiment'] == sentiment]
        
        print(f"\n{sentiment.upper()} REVIEWS:")
        print("-" * 40)
        
        # Show 3 sample reviews
        for i, (_, row) in enumerate(sentiment_reviews.sample(min(3, len(sentiment_reviews))).iterrows()):
            text = row['text'][:200] + "..." if len(row['text']) > 200 else row['text']
            print(f"{i+1}. {text}")
            print()
else:
    print("📝 Sample data not available")

## 6. Word Frequency Analysis

In [ ]:
# Basic word frequency analysis
if 'text' in combined_df.columns:
    import re
    from collections import Counter
    
    print("🔤 WORD FREQUENCY ANALYSIS")
    print("=" * 60)
    
    # Simple tokenization and word counting
    def simple_tokenize(text):
        # Convert to lowercase and extract words
        words = re.findall(r'\b\w+\b', text.lower())
        # Filter out very short words and common stop words
        stop_words = {'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 'of', 'with', 'by', 'is', 'are', 'was', 'were', 'this', 'that', 'it', 'i', 'you', 'he', 'she', 'we', 'they'}
        words = [word for word in words if len(word) > 2 and word not in stop_words]
        return words
    
    # Analyze top words by sentiment
    if 'sentiment' in combined_df.columns:
        for sentiment in combined_df['sentiment'].unique():
            sentiment_texts = combined_df[combined_df['sentiment'] == sentiment]['text']
            
            # Sample for performance
            sample_texts = sentiment_texts.sample(min(1000, len(sentiment_texts))).tolist()
            
            all_words = []
            for text in sample_texts:
                all_words.extend(simple_tokenize(str(text)))
            
            # Get top 15 words
            word_freq = Counter(all_words)
            top_words = word_freq.most_common(15)
            
            print(f"\nTop words in {sentiment.upper()} reviews:")
            for word, count in top_words:
                print(f"  {word}: {count}")
else:
    print("🔤 Text data not available for word frequency analysis")

## 7. Data Insights and Next Steps

In [ ]:
# Summary insights
print("💡 KEY INSIGHTS")
print("=" * 60)

if len(combined_df) > 0:
    print(f"✅ Dataset loaded successfully with {len(combined_df):,} reviews")
    
    if 'sentiment' in combined_df.columns:
        sentiment_balance = combined_df['sentiment'].value_counts(normalize=True)
        min_class = sentiment_balance.min()
        if min_class >= 0.4:
            print(f"✅ Well-balanced dataset (minority class: {min_class:.1%})")
        else:
            print(f"⚠️  Imbalanced dataset (minority class: {min_class:.1%})")
    
    if 'text' in combined_df.columns:
        avg_length = combined_df['text'].str.len().mean()
        if 50 <= avg_length <= 1000:
            print(f"✅ Good text length diversity (avg: {avg_length:.0f} characters)")
        else:
            print(f"⚠️  Consider text length distribution (avg: {avg_length:.0f} characters)")
    
    duplicates_pct = combined_df.duplicated().mean() if len(combined_df) > 1 else 0
    if duplicates_pct < 0.05:
        print(f"✅ Low duplicate rate ({duplicates_pct:.1%})")
    else:
        print(f"⚠️  Consider deduplication ({duplicates_pct:.1%} duplicates)")

print(f"\n🚀 RECOMMENDED NEXT STEPS:")
print(f"1. Run preprocessing: python src/preprocessing.py")
print(f"2. Extract NLP features: python src/feature_extraction.py")
print(f"3. Train baseline models: python src/models.py")
print(f"4. Advanced analysis: notebooks/02_preprocessing.ipynb")
print(f"5. Model training: notebooks/04_model_training.ipynb")

In [ ]:
# Save exploration summary
exploration_summary = {
    'dataset_size': len(combined_df),
    'train_size': len(train_df) if 'train_df' in locals() else 0,
    'test_size': len(test_df) if 'test_df' in locals() else 0,
    'columns': list(combined_df.columns),
    'exploration_completed': True
}

if 'sentiment' in combined_df.columns:
    exploration_summary['sentiment_distribution'] = combined_df['sentiment'].value_counts().to_dict()

if 'text' in combined_df.columns:
    exploration_summary['avg_text_length'] = float(combined_df['text'].str.len().mean())
    exploration_summary['text_length_stats'] = {
        'min': int(combined_df['text'].str.len().min()),
        'max': int(combined_df['text'].str.len().max()),
        'median': float(combined_df['text'].str.len().median())
    }

# Save to results directory
import json
results_dir = PROJECT_ROOT / 'results'
results_dir.mkdir(exist_ok=True)

with open(results_dir / 'data_exploration_summary.json', 'w') as f:
    json.dump(exploration_summary, f, indent=2)

print("💾 Exploration summary saved to results/data_exploration_summary.json")